In [2]:
import numpy as np
import sympy as sym
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.signal import StateSpace, lsim, TransferFunction

# defining symbols

In [34]:
m, g, d, delta, r, R, L0, L1, alpha, c, k, b, phi, tau_m, K_m, i, u = sym.symbols('m, g, d, \\delta, r, R, L0, L1, \\alpha, c, k, b, \\phi, tau_m, K_m, i, u', positive = True)
x1, x2, x3, x4, x_sp = sym.symbols('x1, x2, x3, x4, x_sp', real = True)

# state space representation

In [4]:
x1_dot = x2
x1_dot

x2

In [5]:
x2_dot = (5/(7*m))*(m*g*sym.sin(phi)+(c*(x3**2)/(delta-x1)**2)-k*(x1-d)-b*x2)
x2_dot

5*(-b*x2 + c*x3**2/(\delta - x1)**2 + g*m*sin(\phi) - k*(-d + x1))/(7*m)

In [6]:
x3_dot = (u-x3*R)/(L0+L1*sym.exp(-alpha*(delta-x1)))
x3_dot

(-R*x3 + u)/(L0 + L1*exp(-\alpha*(\delta - x1)))

# Linearisation

$f_1$ already linear

$f_2(x_1, x_2, x_3) = (5/7m)(mgsinϕ + (c(x_3)^2)/(δ-x_1)^2 - k(x_1-d) - bx_2)$

In [7]:
f2 = x2_dot
f2_x1 = sym.diff(f2, x1)
f2_x1

5*(2*c*x3**2/(\delta - x1)**3 - k)/(7*m)

In [8]:
f2_x2 = sym.diff(f2, x2)
f2_x2

-5*b/(7*m)

In [9]:
f2_x3 = sym.diff(f2, x3)
f2_x3

10*c*x3/(7*m*(\delta - x1)**2)

$f_3(x_1, x_3, u) = (u - x_3R)/(L_0 + L_1exp(-α(δ-x_1)))$

In [10]:
f3= x3_dot
f3_x1 = sym.diff(f3, x1)
f3_x1

-L1*\alpha*(-R*x3 + u)*exp(-\alpha*(\delta - x1))/(L0 + L1*exp(-\alpha*(\delta - x1)))**2

In [11]:
f3_x3 = sym.diff(f3, x3)
f3_x3

-R/(L0 + L1*exp(-\alpha*(\delta - x1)))

In [12]:
f3_u = sym.diff(f3, u)
f3_u

1/(L0 + L1*exp(-\alpha*(\delta - x1)))

linearised equations

In [14]:
x1bar, x2bar, x3bar, ubar = sym.symbols('x1bar, x2bar, x3bar, ubar', postive = True)
x1eq, x2eq, x3eq, ueq = sym.symbols('x1eq, x2eq, x3eq, ueq', positive = True)

derivatives at equilibrium point

In [15]:
def evaluate_at_equilibrium(f):
    return f.subs( [ (x1, x1eq), (x2 ,x2eq),  (x3, x3eq), (u, ueq) ])

f2eq = evaluate_at_equilibrium(f2)
f3eq = evaluate_at_equilibrium(f3)

f2_x1eq = evaluate_at_equilibrium(f2_x1)
f2_x3eq = evaluate_at_equilibrium(f2_x3)
f3_x1eq = evaluate_at_equilibrium(f3_x1)
f3_x3eq = evaluate_at_equilibrium(f3_x3)
f3_ueq = evaluate_at_equilibrium(f3_u)

In [16]:
f2eq

5*(-b*x2eq + c*x3eq**2/(\delta - x1eq)**2 + g*m*sin(\phi) - k*(-d + x1eq))/(7*m)

In [17]:
f3eq

(-R*x3eq + ueq)/(L0 + L1*exp(-\alpha*(\delta - x1eq)))

In [18]:
f2_x1eq

5*(2*c*x3eq**2/(\delta - x1eq)**3 - k)/(7*m)

In [19]:
f2_x3eq

10*c*x3eq/(7*m*(\delta - x1eq)**2)

In [20]:
f3_x1eq

-L1*\alpha*(-R*x3eq + ueq)*exp(-\alpha*(\delta - x1eq))/(L0 + L1*exp(-\alpha*(\delta - x1eq)))**2

In [21]:
f3_x3eq

-R/(L0 + L1*exp(-\alpha*(\delta - x1eq)))

In [23]:
f3_ueq

1/(L0 + L1*exp(-\alpha*(\delta - x1eq)))

In [24]:
#deviation variables
#x1bar = x1 - x1eq
#x2bar = x2 - x2eq
#x3bar = x3 - x3eq
#ubar = u - ueq

In [26]:
#f2 linearised
f2lin = f2eq + f2_x1eq*x1bar + f2_x2*x2bar + f2_x3eq*x3bar
f2lin

-5*b*x2bar/(7*m) + 10*c*x3bar*x3eq/(7*m*(\delta - x1eq)**2) + 5*x1bar*(2*c*x3eq**2/(\delta - x1eq)**3 - k)/(7*m) + 5*(-b*x2eq + c*x3eq**2/(\delta - x1eq)**2 + g*m*sin(\phi) - k*(-d + x1eq))/(7*m)

In [27]:
#f3 linearised
f3lin = f3eq + f3_x1eq*x1bar + f3_x3eq*x3bar + f3_ueq*ubar
f3lin

-L1*\alpha*x1bar*(-R*x3eq + ueq)*exp(-\alpha*(\delta - x1eq))/(L0 + L1*exp(-\alpha*(\delta - x1eq)))**2 - R*x3bar/(L0 + L1*exp(-\alpha*(\delta - x1eq))) + ubar/(L0 + L1*exp(-\alpha*(\delta - x1eq))) + (-R*x3eq + ueq)/(L0 + L1*exp(-\alpha*(\delta - x1eq)))

Numerical parameters

In [28]:
#system parameters
m = 0.462
g = 9.81
d = 0.42
delta = 0.65
r = 0.123
R = 2200
L0 = 0.125
L1 = 0.0241
alpha = 1.2
c = 6.811
k = 1885
b = 10.4
phi = np.deg2rad(41)
tau_m = 0.03
K_m = 1

#at equilibrium
i_sp, u_sp = sym.symbols('i_sp, u_sp', positive = True)
x1eq = x_sp = 0.5
x2eq = 0
x3eq = i_sp

working out i_sp

In [29]:
i_sp = np.sqrt((1/c)*((delta - x_sp)**2)*(k*(x_sp-d)-m*g*np.sin(phi)))
print(i_sp)

0.698814821219778


working out u_sp

In [30]:
u_sp = i_sp * R
print(u_sp)

1537.3926066835115


Numerical equilibrium

In [32]:
f2 = (5/(7*m))*(m*g*sym.sin(phi)+(c*(x3**2)/(delta-x1)**2)-k*(x1-d)-b*x2)
f2eq = evaluate_at_equilibrium(f2)

f2_x1 = sym.diff(f2, x1)
f2_x1eq = evaluate_at_equilibrium(f2_x1)

f2_x2 = sym.diff(f2, x2)
f2_x2eq = evaluate_at_equilibrium(f2_x2)

f2_x3 = sym.diff(f2, x3)
f2_x3eq = evaluate_at_equilibrium(f2_x3)

f2lin = f2eq + f2_x1eq*x1bar + f2_x2*x2bar + f2_x3eq*x3bar
f2lin

468.013468013468*i_sp**2 + 936.026936026936*i_sp*x3bar + x1bar*(6240.17957351291*i_sp**2 - 2914.3475572047) - 16.0791589363018*x2bar - 228.550705237521

In [33]:
f3 = (u-x3*R)/(L0+L1*sym.exp(-alpha*(delta-x1)))
f3eq = evaluate_at_equilibrium(f3)

f3_x1 = sym.diff(f3, x1)
f3_x1eq = evaluate_at_equilibrium(f3_x1)

f3_x3 = sym.diff(f3, x3)
f3_x3eq = evaluate_at_equilibrium(f3_x3)

f3_u = sym.diff(f3, u)
f3_ueq = evaluate_at_equilibrium(f3_u)

f3lin = f3eq + f3_x1eq*x1bar + f3_x3eq*x3bar + f3_ueq*ubar
f3lin

-15158.8218607722*i_sp + 6.89037357307828*ubar + 6.89037357307828*ueq + x1bar*(2523.09440065672*i_sp - 1.1468610912076*ueq) - 15158.8218607722*x3bar